In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Veriyi okuma (Colab kullanıyorsan buradaki dosya yolunu kendi Drive yolunla değiştir)
df = pd.read_csv('/content/drive/MyDrive/Buyuk_Veri_Donem_Projesi/twitter_big_data_pipeline/data/processed/cleaned_tweets.csv')

# Tarih formatını düzeltelim (zaman serisi analizi için lazım olacak)
df['tweet_created'] = pd.to_datetime(df['tweet_created'])

print("--- 1. Grafik: Duygu Dağılımı (Sentiment Distribution) ---")
plt.figure(figsize=(8, 8), dpi=150)
duygu_sayilari = df['airline_sentiment'].value_counts()
renkler = ['#ff9999','#66b3ff','#99ff99'] # Negatif, Nötr, Pozitif renkleri
plt.pie(duygu_sayilari, labels=duygu_sayilari.index, autopct='%1.1f%%', startangle=140, colors=renkler)
plt.title('Havayolu Tweetleri: Duygu Dağılımı', fontsize=16)
plt.show()

In [ ]:
print("--- 2. Grafik: En Sık Kullanılan Kelimeler (WordCloud) ---")

# Temizlenmiş tüm tweet metinlerini tek bir dev string (metin) haline getirelim
# Null değerleri atlıyoruz ki hata vermesin
tum_metin = " ".join(tweet for tweet in df['cleaned_text'].dropna())

plt.figure(figsize=(10, 6), dpi=150)
wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(tum_metin)

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off') # Eksenleri gizle
plt.title('Tweetlerde En Çok Geçen Kelimeler', fontsize=16)
plt.show()

In [ ]:
print("--- 3. Grafik: Zaman Serisi Analizi (Günlük Tweet Sayısı) ---")

# Sadece tarih kısmını (saati atarak) alıp günlere göre gruplayalım
df['tarih_gun'] = df['tweet_created'].dt.date
gunluk_tweet = df.groupby('tarih_gun').size()

plt.figure(figsize=(12, 5), dpi=150)
sns.lineplot(x=gunluk_tweet.index, y=gunluk_tweet.values, marker='o', color='b')
plt.title('Zamana Göre Tweet Sayısı Değişimi', fontsize=14)
plt.xlabel('Tarih')
plt.ylabel('Atılan Tweet Sayısı')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
print("--- 4. Grafik: En Aktif 10 Kullanıcı ---")

plt.figure(figsize=(10, 6), dpi=150)
# En çok tweet atan ilk 10 kullanıcı
aktif_kullanicilar = df['name'].value_counts().head(10)

sns.barplot(x=aktif_kullanicilar.values, y=aktif_kullanicilar.index, palette='viridis')
plt.title('En Çok Tweet Atan 10 Kullanıcı', fontsize=14)
plt.xlabel('Tweet Sayısı')
plt.ylabel('Kullanıcı Adı')
plt.show()

In [ ]:
print("--- 5. Grafik: Serbest Seçim (Şikayetlerin Kök Nedenleri) ---")

# Sadece negatif olanları ve "Belirtilmedi" olmayanları filtreleyelim
negatif_sebepler = df[(df['airline_sentiment'] == 'negative') & (df['negativereason'] != 'Belirtilmedi')]

plt.figure(figsize=(12, 6), dpi=150)
sns.countplot(data=negatif_sebepler, y='negativereason', order=negatif_sebepler['negativereason'].value_counts().index, palette='rocket')
plt.title('Negatif Tweetlerin Temel Sebepleri (Kök Neden Analizi)', fontsize=14)
plt.xlabel('Şikayet Sayısı')
plt.ylabel('Şikayet Sebebi')
plt.show()